In [1]:
import yfinance as yf
import pandas as pd
import ssl
import time
from datetime import datetime, timedelta
import requests
import io

# descargo datos desde 2018 para coincidirlo con el indice F&G 
btc = yf.download("BTC-USD", start="2018-02-01", auto_adjust=False)
ssl._create_default_https_context = ssl._create_unverified_context

# reseteo el indice, dejo de tener date como indice y pasa a ser columna
btc.reset_index(inplace=True)

# Filtro solo las columnas necesarias
btc = btc[["Date", "Close", "High", "Low", "Open", "Volume"]]

# guardo csv base
btc.to_csv("../Data/btc.csv", index=False)

btc.tail() ## Inspecciono las ultimas filas del DF para corroborrar que las columnas esten bien


[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
3115,2026-08-13,63402.171875,63926.332031,62799.289062,63404.425781,18765583756
3116,2026-08-14,62975.593750,63551.562500,62487.699219,63402.089844,20280053024
3117,2026-08-15,63024.320312,63119.132812,62850.960938,62974.335938,10033057106
3118,2026-08-16,62818.652344,63310.777344,62648.574219,63023.421875,9078986635
3119,2026-08-18,64140.011719,64487.648438,64166.785156,64487.648438,21063837696


In [2]:
#Indice Fear and Greed -- Mide el miedl del mercado 

url = "https://api.alternative.me/fng/?date_format=%2701%2F01%2F2018%27&format=csv&limit=50000"
response = requests.get(url)

if response.status_code == 200:
    content = response.content.decode('utf-8')
    start = content.find("fng_value")
    csv_data = content[start:]
    # Leer el CSV desde el string
    df = pd.read_csv(io.StringIO(csv_data))
    # Renombrar columnas
    df = df.rename(columns={
        'fng_value': 'date',
        'fng_classification': 'fng_value',
        'date': 'fng_classification'
    })
    # Guardar el nuevo CSV
    df = df.iloc[:-5]
    df.to_csv("../Data/fear_greed.csv", index=False)
    print(df.head(20)) #Muestro los ultimos 20 datos 
else:
    print("Error:", response.status_code)

          date  fng_value fng_classification
0   18-08-2026       41.0               Fear
1   17-08-2026       31.0               Fear
2   16-08-2026       34.0               Fear
3   15-08-2026       34.0               Fear
4   14-08-2026       29.0               Fear
5   13-08-2026       29.0               Fear
6   12-08-2026       27.0               Fear
7   11-08-2026       29.0               Fear
8   10-08-2026       30.0               Fear
9   09-08-2026       31.0               Fear
10  08-08-2026       30.0               Fear
11  07-08-2026       29.0               Fear
12  06-08-2026       25.0       Extreme Fear
13  05-08-2026       27.0               Fear
14  04-08-2026       25.0       Extreme Fear
15  03-08-2026       28.0               Fear
16  02-08-2026       27.0               Fear
17  01-08-2026       27.0               Fear
18  31-07-2026       25.0       Extreme Fear
19  30-07-2026       28.0               Fear
